# 05 - GenAIServer: one HTTP server for LLM, VLM, and ASR

`pyneat.genai.GenAIServer` is an **in-process, OpenAI-compatible HTTP server**. You add one or more
models to it, `start()` it, and any OpenAI-style client (or plain `curl`) can hit it. This notebook
runs a **two-model** server end to end and exercises all three endpoints:

| Endpoint | Method | What it does |
| --- | --- | --- |
| `/v1/models` | GET | list served model names — the quickest smoke check |
| `/v1/chat/completions` | POST | LLM text chat **and** VLM image+text chat |
| `/v1/audio/transcriptions` | POST (multipart) | ASR transcription |

We serve two models that are already on the board, under short **served names**:

| Served name | Model | Handles |
| --- | --- | --- |
| `vlm` | `Qwen3-VL-4B-Instruct-GPTQ-a16w4` | text chat + image+text chat |
| `asr` | `whisper-small-a16w8` | audio transcription |

A VLM already does text-only chat, so `vlm` covers both chat cases; transcription needs the ASR
model. **These cells run a real server and load ~2 models into memory — run them on the DevKit.**

## Two different things are called "server" — don't confuse them

- **`pyneat.genai.GenAIServer`** (this notebook) — an **in-process** server you embed in your own
  Python. `start()` runs it on a background thread and returns, so the notebook kernel stays free to
  act as the client. This is the application server.
- **`llima run --mode web`** and **`llima benchmark-server`** — separate **CLI processes** you start
  in a terminal. `--mode web` is also OpenAI-compatible HTTP; `benchmark-server` is a ZeroMQ
  benchmarking server. They are not driven from a notebook cell.

All of them default to **port 9998**, so only one can bind it at a time.

## The API surface (and the one rule that matters)

```text
opts   = pyneat.genai.GenAIServerOptions()   # host "0.0.0.0", port 9998
server = pyneat.genai.GenAIServer(opts)
name   = server.add_model(model_dir, served_name)   # register a model under a name
server.model_names()                                # -> ["vlm", "asr", ...]
server.start()    # NON-BLOCKING: background thread, returns immediately
server.stop()     # stop the background server, free the models
```

**Use `start()`, never `serve()`.** `serve()` runs the HTTP loop in the calling thread and would
**block the kernel forever** — the cell never returns. `start()` spawns a background thread (and
releases the GIL), so the cell returns at once and later cells act as the client.

Each request selects its model by the **served name** in the `"model"` field. An unknown name returns
HTTP 404, so register the names first and pass them exactly.

## Step 1 - Start the two-model server

Constructing the server and calling `add_model()` + `start()` **loads the models into memory** (a few
GB, ~seconds each). Keep a reference to `server` alive: if it is garbage-collected, its destructor
stops the server.

In [ ]:
import pyneat as neat

VLM_DIR = "/media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4"
ASR_DIR = "/media/nvme/llima/models/whisper-small-a16w8"

opts = neat.genai.GenAIServerOptions()      # host "0.0.0.0", port 9998
server = neat.genai.GenAIServer(opts)

server.add_model(VLM_DIR, "vlm")            # text chat + image+text chat
server.add_model(ASR_DIR, "asr")            # audio transcription
print("registered:", server.model_names())

server.start()                              # non-blocking; returns immediately
print("server started on http://0.0.0.0:9998")

**Interpretation.** Both models are now resident and the HTTP server is listening on a background
thread. The kernel is free, so the next cells can call the server as an ordinary HTTP client.

## Step 2 - `GET /v1/models` (the smoke check)

The cheapest confirmation that the server is up and the registry is populated. No model runs.

In [ ]:
import requests

BASE = "http://localhost:9998"

r = requests.get(f"{BASE}/v1/models", timeout=10)
r.raise_for_status()
print(r.json())                             # {"object":"list","data":[{"id":"vlm",...},{"id":"asr",...}]}
print("served names:", [m["id"] for m in r.json()["data"]])

## Step 3 - Text prompt in, text out (`POST /v1/chat/completions`)

Plain OpenAI chat: a `messages` list with a string `content`. We route to `"vlm"` (a VLM answers
text-only prompts fine). Omitting `stream` returns a single JSON response.

In [ ]:
payload = {
    "model": "vlm",
    "messages": [{"role": "user", "content": "Give me three tips for designing a small REST API."}],
    "max_tokens": 128,
}
r = requests.post(f"{BASE}/v1/chat/completions", json=payload, timeout=120)
r.raise_for_status()
out = r.json()
print(out["choices"][0]["message"]["content"])
print("--- completion_tokens:", out.get("usage", {}).get("completion_tokens"))

## Step 4 - Text + image in, text out (`POST /v1/chat/completions`)

The OpenAI **vision** format: `content` becomes a list mixing a `text` part and an `image_url` part,
where the image is a **base64 `data:` URL**. Same endpoint, same `"vlm"` model. The server decodes
the image and feeds it to the vision encoder (no manual RGB conversion needed — the data URL carries
the encoded bytes).

In [ ]:
import base64, mimetypes

def image_data_url(path: str) -> str:
    mime = mimetypes.guess_type(path)[0] or "image/jpeg"
    b64 = base64.b64encode(open(path, "rb").read()).decode("ascii")
    return f"data:{mime};base64,{b64}"

IMAGE_PATH = "../../tutorial/assets/images/image.png"

payload = {
    "model": "vlm",
    "messages": [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe this image in one sentence."},
            {"type": "image_url", "image_url": {"url": image_data_url(IMAGE_PATH)}},
        ],
    }],
    "max_tokens": 128,
}
r = requests.post(f"{BASE}/v1/chat/completions", json=payload, timeout=120)
r.raise_for_status()
print(r.json()["choices"][0]["message"]["content"])

## Step 5 - Audio file in, transcript out (`POST /v1/audio/transcriptions`)

Transcription is a **multipart** request, not JSON: form fields (`model`, `language`) plus a `file`
upload. It routes to the `"asr"` model — the server rejects it with HTTP 400 if the named model is
not an ASR model. The non-streaming response is `{"text": ...}`.

In [ ]:
AUDIO_PATH = "../02-run-llm-vlm/assets/audio.wav"

with open(AUDIO_PATH, "rb") as audio:
    r = requests.post(
        f"{BASE}/v1/audio/transcriptions",
        data={"model": "asr", "language": "en"},
        files={"file": ("audio.wav", audio, "audio/wav")},
        timeout=120,
    )
r.raise_for_status()
print("transcript:", r.json()["text"])

## Step 6 - Stop the server

`stop()` shuts down the background thread and frees both models from memory. Always stop the server
when done (or let the `server` object be deleted, which stops it too) so the MLA is released.

In [ ]:
server.stop()
print("server stopped")

## The same three calls as `curl`

Handy for testing from any shell, or from a different machine (replace `localhost` with the board
host). The server sets permissive CORS, so a browser client works too.

```bash
# list served names
curl -s http://localhost:9998/v1/models

# text chat
curl -s http://localhost:9998/v1/chat/completions -H 'Content-Type: application/json' \
  -d '{"model":"vlm","messages":[{"role":"user","content":"Say hello."}],"max_tokens":64}'

# audio transcription (multipart)
curl -s http://localhost:9998/v1/audio/transcriptions \
  -F model=asr -F language=en -F file=@../02-run-llm-vlm/assets/audio.wav
```

(Image chat via curl works the same as the JSON above, with the base64 `data:` URL inline — it is
long, so the Python helper above is the practical way to build it.)

## Expected output

- **`/v1/models`** → `{"object": "list", "data": [{"id": "vlm", ...}, {"id": "asr", ...}]}`.
- **Text chat** → a few sentences of REST-API advice, plus a `completion_tokens` count.
- **Image + text** → one sentence describing the picture.
- **Audio** → the transcript string of `audio.wav`.

If a chat call returns HTTP **404 "Unknown model"**, the `"model"` name does not match a registered
served name — check `server.model_names()`. If audio returns **400 "not an ASR model"**, you pointed
it at `"vlm"` instead of `"asr"`. If `start()` throws that the port is in use, another server
(a previous cell, or `llima run --mode web`) already holds **9998** — stop it first.

## Streaming (optional)

Every chat/audio call above is non-streaming for simplicity. To stream tokens as they generate, add
`"stream": True` to a chat payload (or `stream="true"` to the audio form) and read the response as
Server-Sent Events: lines beginning `data: `, each a JSON chunk with `choices[0].delta.content`
(chat) or `text` (audio), terminated by `data: [DONE]`. See the streaming clients in
[`021_serve_genai_models`](https://github.com/sima-neat/core/tree/main/tutorials/021_serve_genai_models).

## References

- [`GenAIServer.h`](https://github.com/sima-neat/core/blob/main/include/genai/GenAIServer.h) — `GenAIServerOptions`, `add_model`, `model_names`, `start`, `stop`
- [`GenAIServer.cpp`](https://github.com/sima-neat/core/blob/main/src/genai/GenAIServer.cpp) — the route handlers and request/response shapes
- [`module.cpp`](https://github.com/sima-neat/core/blob/main/python/src/module.cpp) — the `pyneat.genai.GenAIServer` bindings (`start` releases the GIL)
- [`021_serve_genai_models`](https://github.com/sima-neat/core/tree/main/tutorials/021_serve_genai_models) — the reference `request_*` clients (text / image / audio), streaming
- `../02-run-llm-vlm/` — running the same models directly, without a server